In [ ]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

In [ ]:

# LangSmith tracing
os.environ["LANGSMITH_TRACING"] = os.getenv("LANGSMITH_TRACING", "true")
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY", "")
os.environ["LANGSMITH_PROJECT"] = os.getenv("LANGSMITH_PROJECT", "simple-rag-app")

In [ ]:
# 1. Load PDF
loader = PyPDFLoader("data/sample.pdf")
docs = loader.load()

In [ ]:
# 2. Split documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
splits = text_splitter.split_documents(docs)

In [ ]:
# 3. Create embeddings + FAISS vector DB
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(splits, embeddings)

In [ ]:
# 4. Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
# 5. Create LLM
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

In [ ]:
# 6. Prompt
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Answer the question only using the given context.

Context:
{context}

Question:
{question}
""")

In [ ]:

# 7. Ask questions
while True:
    question = input("\nAsk question: ")

    if question.lower() in ["exit", "quit"]:
        break

    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    chain = prompt | llm
    answer = chain.invoke({
        "context": context,
        "question": question
    })

    print("\nAnswer:")
    print(answer.content)